In [5]:
import Meshes as Meshes
using CairoMakie: CairoMakie as Makie
using Unitful: m, ustrip, Length
using LinearAlgebra: cross, dot, norm
using MarineHydro

using FiniteDifferences
using DifferentiationInterface 
import ForwardDiff 
import Zygote
import Enzyme
import Mooncake


In [6]:
r1 = 0.88
r2 = 0.35
d1 = 0.16
d2 = 0.37
show_plot = true

mhmesh = wavebot_mesh(r1,r2,d1,d2)

Mesh([0.88 0.0 -0.0; 0.876651334320736 0.07669705361793919 -0.0; … ; 0.08716703608302773 -0.007626127490420102 -0.53; 0.0 0.0 -0.53], [1 73 74 2; 2 74 75 3; … ; 1223 1225 1224 -1; 1224 1225 1153 -1], Number[0.8783256671603681 0.03834852680896959 -0.020000000000000004; 0.8716410784857396 0.11475372498241895 -0.020000000000000004; … ; 0.05777923815719863 -0.00760678101209217 -0.53; 0.05822234536100924 -0.002542042496806701 -0.53], Number[0.9990482215818576 0.04361938736533626 0.0; 0.9914448613738104 0.13052619222005138 0.0; … ; 0.0 0.0 -1.0; 0.0 0.0 -1.0], [0.0030708048705196545, 0.003070804870519654, 0.0030708048705196545, 0.0030708048705196537, 0.0030708048705196554, 0.0030708048705196532, 0.003070804870519656, 0.0030708048705196515, 0.0030708048705196545, 0.0030708048705196576  …  0.00033364307770587847, 0.00033364307770587825, 0.0003336430777058785, 0.000333643077705882, 0.00033364307770587527, 0.0003336430777058817, 0.00033364307770587884, 0.0003336430777058783, 0.000333643077705878

In [4]:
function get_area(x)
    mhmesh = wavebot_mesh(x[1],x[2],x[3],x[4])
    return sum(mhmesh.areas)
end

backend = AutoForwardDiff()
x_val = [0.88, 0.35, 0.16, 0.37]

# AD
AD_grad = gradient(get_area, backend, x_val)
display(AD_grad)

# FD
fdm = central_fdm(5, 1)
FD_grad = FiniteDifferences.grad(fdm, get_area, x_val)[1]
display(FD_grad)

4-element Vector{Float64}:
 6.197133540022757
 1.0615864544540885
 5.527448766935356
 2.2126495352470887

4-element Vector{Float64}:
 6.197133540024523
 1.061586454457087
 5.5274487669355095
 2.212649535245132

In [7]:
function axisymmetric_mesh(profile::Meshes.Rope, n_theta::Integer=72)
    @assert n_theta >= 3 "n_theta must be at least 3"
    @assert length(profile.vertices) >= 2 "profile must contain at least two points"

    # Points: Revolve each 2D profile point (r, z) into a 3D ring,
    #  treat zero radius points as special cases
    p1 = Meshes.coords(first(profile.vertices))
    points = typeof(Meshes.Point(p1.x, zero(p1.x), p1.y))[]
    ring_starts = Int[]
    ring_sizes = Int[]

    for p in profile.vertices
        r = Meshes.coords(p).x
        z = Meshes.coords(p).y
        @assert r >= zero(r) "profile radii must be nonnegative"
        push!(ring_starts, length(points) + 1)
        if iszero(r)
            push!(ring_sizes, 1)
            push!(points, Meshes.Point(zero(r), zero(r), z))
        else
            push!(ring_sizes, n_theta)
            for i in 0:(n_theta-1)
                θ = 2pi * i / n_theta
                push!(points, Meshes.Point(r * cos(θ), r * sin(θ), z))
            end
        end
    end

    # Connections: use quads between regular rings and triangles for zero radius points.
    connec = Union{
        typeof(Meshes.connect((1, 2, 3), Meshes.Triangle)),
        typeof(Meshes.connect((1, 2, 3, 4), Meshes.Quadrangle)),
    }[]
    for layer in 1:(length(profile.vertices)-1)
        start_1 = ring_starts[layer]
        start_2 = ring_starts[layer+1]
        size_1 = ring_sizes[layer]
        size_2 = ring_sizes[layer+1]

        if size_1 == 1 && size_2 == 1
            error("adjacent points cannot both have radius 0")
        elseif size_1 == 1
            ir0 = start_1
            for i in 0:(n_theta-1)
                i3 = start_2 + i
                i4 = start_2 + mod(i + 1, n_theta)
                push!(connec, Meshes.connect((ir0, i3, i4), Triangle))
            end
        elseif size_2 == 1
            ir0 = start_2
            for i in 0:(n_theta-1)
                i1 = start_1 + i
                i2 = start_1 + mod(i + 1, n_theta)
                push!(connec, Meshes.connect((i1, ir0, i2), Meshes.Triangle))
            end
        else
            for i in 0:(n_theta-1)
                i1 = start_1 + i
                i2 = start_1 + mod(i + 1, n_theta)
                i3 = start_2 + i
                i4 = start_2 + mod(i + 1, n_theta)
                push!(connec, Meshes.connect((i1, i3, i4, i2), Meshes.Quadrangle))
            end
        end
    end

    return Meshes.SimpleMesh(points, connec)
end

function wavebot_profile(
    cylinder_height::Union{Length,Number}=0.16m,
    frustum_height::Union{Length,Number}=0.37m,
    top_radius::Union{Length,Number}=0.88m,
    bottom_radius::Union{Length,Number}=0.35m;
    n_points::NTuple{3,Integer}=(5, 10, 5),
)
    R = promote_type(typeof(top_radius), typeof(bottom_radius))
    Z = promote_type(typeof(cylinder_height), typeof(bottom_radius))
    profile_points = typeof(Meshes.Point(zero(R), zero(Z)))[]

    n_bottom, n_frustum, n_cylinder = n_points

    for j in 0:(n_cylinder-1)
        t = j / (n_cylinder - 1)
        z = -cylinder_height * t
        push!(profile_points, Meshes.Point(top_radius, z))
    end

    for j in 1:(n_frustum-1)
        t = j / (n_frustum - 1)
        radius = top_radius + (bottom_radius - top_radius) * t
        z = -cylinder_height - frustum_height * t
        push!(profile_points, Meshes.Point(radius, z))
    end

    z_bottom = -cylinder_height - frustum_height
    for j in 1:(n_bottom-1)
        t = j / (n_bottom - 1)
        push!(profile_points, Meshes.Point(bottom_radius * (1 - t), z_bottom))
    end

    return Meshes.Rope(profile_points)
end



wavebot_profile (generic function with 5 methods)

In [9]:
r1 = 0.88
r2 = 0.35
d1 = 0.16
d2 = 0.37

cylinder_height=d1
frustum_height=d2
top_radius=r1
bottom_radius=r2

# mesh params
n_points = (5, 10, 5)
n_theta = 72

profile = wavebot_profile(cylinder_height, frustum_height, top_radius, bottom_radius; n_points=n_points)
mesh = axisymmetric_mesh(profile, n_theta)

1224 SimpleMesh
  1225 vertices
  ├─ Point(x: 0.88 m, y: 0.0 m, z: -0.0 m)
  ├─ Point(x: 0.876651334320736 m, y: 0.07669705361793919 m, z: -0.0 m)
  ├─ Point(x: 0.8666308226507431 m, y: 0.1528103963468987 m, z: -0.0 m)
  ├─ Point(x: 0.8500147271343801 m, y: 0.22776075969021825 m, z: -0.0 m)
  ├─ Point(x: 0.8269295062915994 m, y: 0.30097772612658846 m, z: -0.0 m)
  ⋮
  ├─ Point(x: 0.08222310431876698 m, y: -0.029926762540996 m, z: -0.53 m)
  ├─ Point(x: 0.08451850980029348 m, y: -0.022646666446470558 m, z: -0.53 m)
  ├─ Point(x: 0.08617067838856819 m, y: -0.015194215545856407 m, z: -0.53 m)
  ├─ Point(x: 0.08716703608302773 m, y: -0.007626127490420102 m, z: -0.53 m)
  └─ Point(x: 0.0 m, y: 0.0 m, z: -0.53 m)
  1224 elements
  ├─ Quadrangle(1, 73, 74, 2)
  ├─ Quadrangle(2, 74, 75, 3)
  ├─ Quadrangle(3, 75, 76, 4)
  ├─ Quadrangle(4, 76, 77, 5)
  ├─ Quadrangle(5, 77, 78, 6)
  ⋮
  ├─ Triangle(1220, 1225, 1221)
  ├─ Triangle(1221, 1225, 1222)
  ├─ Triangle(1222, 1225, 1223)
  ├─ Triangle(122

In [15]:
a = (1,2,3)
collect(a)

3-element Vector{Int64}:
 1
 2
 3

In [19]:
# function Mesh(mesh::Meshes.SimpleMesh)

# get vertices
verts = mesh.vertices
x_vals = [ustrip.(v.coords.x) for v in verts]
y_vals = [ustrip.(v.coords.y) for v in verts]
z_vals = [ustrip.(v.coords.z) for v in verts]
vertices_mat = hcat(x_vals,y_vals,z_vals)

# get faces
connec = Meshes.topology(mesh).connec
n_faces = length(connec)
faces_mat = zeros(Number, n_faces, 4)
for i in 1:n_faces
    inds = Meshes.indices(connec[i])
    inds_vec = collect(inds)
    if length(inds)==3 # triangle element (repeat last vertex like cpt)
        faces_mat[i, :] = vcat(inds_vec,inds_vec[3]) 
    else
        faces_mat[i, :] = inds_vec
    end
    # faces_mat[i, 1:length(inds)] .= inds
end



# get areas
areas_vec = ustrip.(Meshes.area.(mesh))

# get nvertices
nvertices = length(vertices_mat[:,1])

# get nfaces 
nfaces = length(faces_mat[:,1])

# get centers and normals (area centroid, not verex centroid)
center_mat = zeros(Number, nfaces,3)
normals_mat = zeros(Number, nfaces,3)
radii_vec = zeros(Number, nfaces)
for i in 1:nfaces

    # compute center
    faces_vec = faces_mat[i,:]
    if faces_vec[end]==-1 # this is a triangle element
        a = vertices_mat[faces_vec[1],:]
        b = vertices_mat[faces_vec[2],:]
        c = vertices_mat[faces_vec[3],:]
        center_vec = (a+b+c)/3
    else # this is a quadrilateral element
        a = vertices_mat[faces_vec[1],:]
        b = vertices_mat[faces_vec[2],:]
        c = vertices_mat[faces_vec[3],:]
        d = vertices_mat[faces_vec[4],:]
        area1 = 0.5 * norm(cross(b-a, c-a))
        area2 = 0.5 * norm(cross(c-a, d-a))
        c1 = (a+b+c)/3
        c2 = (a+c+d)/3
        center_vec = (c1*area1 + c2*area2) / (area1 + area2)
    end
    center_mat[i,:] = center_vec

    # compute normal
    normals_vec = cross(b-a, c-a)# not yet unit length
    normals_mat[i,:] = normals_vec/norm(normals_vec) # normalize

    # compute radii
    radii_vec[i] = norm(a-center_vec) # distance between center and first vertex
end

#     return Mesh(vertices_mat, faces_mat, center_mat, normals_mat, areas_vec, radii_vec, nvertices, nfaces)
# end      




In [79]:
function plot_mesh(mesh::Meshes.SimpleMesh, profile)

    profile_r = [ustrip.(v.coords.x) for v in profile.vertices]
    profile_z = [ustrip.(v.coords.y) for v in profile.vertices]

    fig = Makie.Figure(size=(1200, 600));
    ax = Makie.Axis(
        fig[1, 1];
        aspect=Makie.DataAspect(),
        title="WaveBot Profile",
        xlabel="r [m]",
        ylabel="z [m]",
        limits=((0, nothing), (nothing, 1e-6)),
    );
    Makie.lines!(ax, profile_r, profile_z, color=:gold, linewidth=2);
    Makie.scatter!(ax, profile_r, profile_z, color=:black, markersize=10);

    ax = Makie.Axis3(
        fig[1, 2];
        aspect=:data,
        azimuth=π / 8,
        elevation=π / 8,
        title="WaveBot Mesh",
        xlabel="x [m]",
        ylabel="y [m]",
        zlabel="z [m]",
    );
    Meshes.viz!(ax, mesh, color=:gold, showsegments=true, segmentcolor=:grey50, segmentsize=0.25);

    display(fig)
end

plot_mesh (generic function with 2 methods)

In [91]:
# Example: Create mesh & plot

function wavebot_mesh(r1,r2,d1,d2,show_plot=false)

    # geometric design variables
    cylinder_height=d1
    frustum_height=d2
    top_radius=r1
    bottom_radius=r2

    # mesh params
    n_points = (5, 10, 5)
    n_theta = 72

    profile = wavebot_profile(cylinder_height, frustum_height, top_radius, bottom_radius; n_points=n_points)
    mesh = axisymmetric_mesh(profile, n_theta)
    MHmesh = Mesh(mesh)

    if show_plot
        plot_mesh(mesh, profile)
    end
        
    return MHmesh
end

wavebot_mesh (generic function with 3 methods)

In [92]:
r1 = 0.88
r2 = 0.35
d1 = 0.16
d2 = 0.37
show_plot = true

mhmesh = wavebot_mesh(r1,r2,d1,d2)

Mesh([0.88 0.0 -0.0; 0.876651334320736 0.07669705361793919 -0.0; … ; 0.08716703608302773 -0.007626127490420102 -0.53; 0.0 0.0 -0.53], [1 73 74 2; 2 74 75 3; … ; 1223 1225 1224 -1; 1224 1225 1153 -1], Number[0.8783256671603681 0.03834852680896959 -0.020000000000000004; 0.8716410784857396 0.11475372498241895 -0.020000000000000004; … ; 0.05777923815719863 -0.00760678101209217 -0.53; 0.05822234536100924 -0.002542042496806701 -0.53], Number[0.9990482215818576 0.04361938736533626 0.0; 0.9914448613738104 0.13052619222005138 0.0; … ; 0.0 0.0 -1.0; 0.0 0.0 -1.0], [0.0030708048705196545, 0.003070804870519654, 0.0030708048705196545, 0.0030708048705196537, 0.0030708048705196554, 0.0030708048705196532, 0.003070804870519656, 0.0030708048705196515, 0.0030708048705196545, 0.0030708048705196576  …  0.00033364307770587847, 0.00033364307770587825, 0.0003336430777058785, 0.000333643077705882, 0.00033364307770587527, 0.0003336430777058817, 0.00033364307770587884, 0.0003336430777058783, 0.000333643077705878

In [82]:
function get_area(x)
    mhmesh = wavebot_mesh(x[1],x[2],x[3],x[4])
    return sum(mhmesh.areas)
end

get_area (generic function with 2 methods)

In [90]:
# verify differentiablity. Yes, it works! (next time, I should be careful with initiating zeros with type Number to avoid issues with ForwardDiff's dual numbers)

backend = AutoForwardDiff()
x_val = [0.88, 0.35, 0.16, 0.37]

# AD
AD_grad = gradient(get_area, backend, x_val)
display(AD_grad)

# FD
fdm = central_fdm(5, 1)
FD_grad = FiniteDifferences.grad(fdm, get_area, x_val)[1]
display(FD_grad)



4-element Vector{Float64}:
 6.197133540022757
 1.0615864544540885
 5.527448766935356
 2.2126495352470887

4-element Vector{Float64}:
 6.197133540024523
 1.061586454457087
 5.5274487669355095
 2.212649535245132